In [ ]:
%load_ext watermark


In [ ]:
import os
import subprocess

os.environ["POLARS_FORCE_NEW_STREAMING"] = "1"

import pandas as pd
import polars as pl
from tqdm import tqdm

from pylib._seed_global_rngs import seed_global_rngs


In [ ]:
pd.options.display.float_format = "{:,.1f}".format


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = "2025-05-30-compscreen-mutcount"
teeplot_subdir


In [ ]:
seed_global_rngs(1)


## Get Data


In [ ]:
data_sources = {
    "uk": "https://osf.io/mkjy5/download",
    "multistrain": "https://osf.io/ywmpt/download",
    "vanilla": "https://osf.io/r8skg/download",
    "vanilla-big": "https://osf.io/j4795/download",
    "vanilla-big-1.3x": "https://osf.io/cnp5z/download",
}
tmp_path = f"/tmp/{teeplot_subdir}.pqt"


In [ ]:
results = []

for source_name, url in data_sources.items():
    print(f"Downloading {source_name} data from {url}")

    subprocess.run(
        [
            "wget",
            "--tries=5",
            "--show-progress",
            "--progress=bar:force",
            "-O",
            str(tmp_path),
            url,
        ],
        check=True,
    )
    print("done!")

    df = pl.scan_parquet(
        tmp_path,
        low_memory=True,
        retries=5,
    )

    unique_groups = (
        df.unique(
            [
                "trt_name",
                "trt_n_downsample",
                "trt_hsurf_bits",
                "replicate_uuid",
            ]
        )
        .select(
            pl.col("trt_name"),
            pl.col("trt_n_downsample"),
            pl.col("trt_hsurf_bits"),
            pl.col("replicate_uuid"),
        )
        .drop_nans()
        .drop_nulls()
        .collect(engine="streaming")
    )

    for (trt_name, trt_n_downsample, trt_hsurf_bits, replicate_uuid) in tqdm(
        [*unique_groups.iter_rows()],
    ):
        group = df.filter(
            (pl.col("trt_name") == trt_name)
            & (pl.col("trt_n_downsample") == trt_n_downsample)
            & (pl.col("trt_hsurf_bits") == trt_hsurf_bits)
            & (pl.col("replicate_uuid") == replicate_uuid)
        ).collect(engine="streaming")

        group_df = group.to_pandas()
        res = [
            {
                "sum leaf count": group_df.loc[
                    group_df["is_focal_defmut"],
                    "num_leaves",
                ].sum(),
                "sum defmut": group_df["is_focal_defmut"].astype(bool).sum(),
            },
        ]
        results.extend(
            {
                "source_name": source_name,
                "trt_name": trt_name,
                "trt_n_downsample": trt_n_downsample,
                "trt_hsurf_bits": trt_hsurf_bits,
                "replicate_uuid": replicate_uuid,
                **record,
            }
            for record in res
        )


In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{teeplot_subdir}-raw.csv",
    index=False,
)
results_df


In [ ]:
summary_df = results_df.groupby(
    ["source_name", "trt_name", "trt_n_downsample", "trt_hsurf_bits"]
).agg(
    {
        "sum leaf count": ["mean", "std"],
        "sum defmut": ["mean", "std"],
    },
)
summary_df.to_csv(
    f"{teeplot_subdir}-summary.csv",
    index=True,
)
summary_df
